# Formatador de Texto com LaTeX → DOCX

Este notebook recebe um texto com fórmulas LaTeX e gera um `.docx` formatado:

- **Fórmulas em bloco** (`$$...$$`) → parágrafo separado, fonte monoespaçada (Courier New), centralizado
- **Fórmulas inline** (`$...$`) → fonte monoespaçada dentro do parágrafo
- **Trechos entre aspas** (`"..."` e `'...'`) → itálico
- **Texto normal** → fonte padrão (Calibri)

### Tratamento de quebras de linha
- `\n` simples (soft wrap) → substituído por espaço (continua no mesmo parágrafo)
- `\n\n` ou mais (parágrafo real) → novo parágrafo no DOCX

### Modos de entrada
- **Opção 1** — Ler de um arquivo `.txt`
- **Opção 2** — Colar/escrever a string diretamente no código

## 1. Instalação de dependências

In [ ]:
!pip install python-docx --quiet

## 2. Configurações gerais

In [ ]:
# ============================================================
# MODO DE ENTRADA
# ============================================================
# Escolha UMA das opções:
#   "arquivo"  → lê de um arquivo .txt
#   "string"   → usa o texto colado diretamente abaixo

MODO = "arquivo"   # <<< ALTERE AQUI: "arquivo" ou "string"

# ============================================================
# Caminho do arquivo (usado apenas se MODO = "arquivo")
# ============================================================

ARQUIVO_ENTRADA = "exemplo.txt"

# ============================================================
# Texto direto (usado apenas se MODO = "string")
# Cole o texto entre as aspas triplas abaixo
# ============================================================

TEXTO_DIRETO = r"""
Cole ou escreva seu texto aqui.
Fórmulas em bloco usam $$...$$, por exemplo: $$x^2 + y^2 = z^2$$
Fórmulas inline usam $...$, por exemplo: $n!$
"""

# ============================================================
# Arquivo de saída
# ============================================================

ARQUIVO_SAIDA = "saida.docx"

# ============================================================
# Configurações de fonte (ajuste se necessário)
# ============================================================

FONTE_TEXTO      = "Calibri"         # Fonte do texto normal
FONTE_FORMULA    = "Courier New"     # Fonte das fórmulas (monoespaçada)
TAMANHO_TEXTO    = 12                # Tamanho em pt do texto normal
TAMANHO_FORMULA  = 11                # Tamanho em pt das fórmulas

## 3. Funções de processamento

In [ ]:
import re
from docx import Document
from docx.shared import Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH


def obter_texto(modo: str, caminho: str = "", texto_direto: str = "") -> str:
    """
    Retorna o texto de entrada conforme o modo escolhido.
      - 'arquivo' : lê do arquivo .txt
      - 'string'  : usa o texto passado diretamente
    """
    if modo == "arquivo":
        with open(caminho, "r", encoding="utf-8") as f:
            texto = f.read()
        print(f"[MODO ARQUIVO] Lido: {caminho} ({len(texto)} caracteres)")
        return texto
    elif modo == "string":
        texto = texto_direto.strip()
        print(f"[MODO STRING] Texto direto ({len(texto)} caracteres)")
        return texto
    else:
        raise ValueError(f"MODO inválido: '{modo}'. Use 'arquivo' ou 'string'.")


def normalizar_quebras(texto: str) -> str:
    """
    Normaliza quebras de linha e caracteres de formatação do texto:
      - Remove barras invertidas antes de aspas (\" → " e \' → ')
      - Remove asteriscos de formatação (*texto* → texto)
      - Preserva quebras de parágrafo (\n\n ou mais) como marcador especial
      - Substitui \n simples (soft wrap) por espaço
      - Restaura marcadores de parágrafo
      - Remove espaços duplicados resultantes
    """
    # 1. Remover barras invertidas antes de aspas (\" → " e \' → ')
    texto = texto.replace('\\"', '"')
    texto = texto.replace("\\'", "'")

    # 2. Remover asteriscos de formatação (*texto* → texto)
    #    Preserva asteriscos dentro de fórmulas LaTeX ($..$ e $$..$$)
    texto = re.sub(r'(?<!\$)\*(?!\*)', '', texto)

    # 3. Marcar quebras de parágrafo reais (\n\n+)
    texto = re.sub(r'\n{2,}', '<<PARAGRAFO>>', texto)

    # 4. Substituir \n simples por espaço (soft wrap)
    texto = texto.replace('\n', ' ')

    # 5. Restaurar quebras de parágrafo
    texto = texto.replace('<<PARAGRAFO>>', '\n\n')

    # 6. Limpar espaços duplicados
    texto = re.sub(r' {2,}', ' ', texto)

    return texto.strip()


def tokenizar(texto: str) -> list:
    """
    Divide o texto em tokens classificados por tipo:
      - 'bloco'   : fórmula em bloco ($$...$$)
      - 'inline'  : fórmula inline ($...$)
      - 'italico' : trecho entre aspas
      - 'texto'   : texto normal
    """
    padrao = re.compile(
        r'(\$\$.+?\$\$)'        # grupo 1: bloco $$...$$
        r'|(\$.+?\$)'            # grupo 2: inline $...$
        r'|(\"[^\"]+?\")'     # grupo 3: aspas duplas
        r"|(\'[^\']+?\')",     # grupo 4: aspas simples
        flags=re.DOTALL
    )

    tokens = []
    pos = 0

    for match in padrao.finditer(texto):
        inicio = match.start()
        if inicio > pos:
            tokens.append(("texto", texto[pos:inicio]))

        if match.group(1):
            tokens.append(("bloco", match.group(1).strip()))
        elif match.group(2):
            tokens.append(("inline", match.group(2).strip()))
        elif match.group(3):
            tokens.append(("italico", match.group(3)))
        elif match.group(4):
            tokens.append(("italico", match.group(4)))

        pos = match.end()

    if pos < len(texto):
        tokens.append(("texto", texto[pos:]))

    return tokens


def adicionar_run(paragrafo, texto, fonte, tamanho, italico=False, cor=None):
    """Adiciona um run formatado a um parágrafo."""
    run = paragrafo.add_run(texto)
    run.font.name = fonte
    run.font.size = Pt(tamanho)
    run.font.italic = italico
    if cor:
        run.font.color.rgb = cor
    return run


print("Funções carregadas com sucesso.")

## 4. Leitura, normalização e tokenização

In [ ]:
# Obter texto conforme o MODO escolhido
texto_bruto = obter_texto(MODO, ARQUIVO_ENTRADA, TEXTO_DIRETO)

# Normalizar quebras de linha (soft wrap → espaço)
texto_normalizado = normalizar_quebras(texto_bruto)

print(f"\n--- Texto normalizado (preview) ---")
print(texto_normalizado[:300] + "...")

# Tokenizar
tokens = tokenizar(texto_normalizado)

print(f"\nTokens encontrados: {len(tokens)}")
for tipo, conteudo in tokens:
    preview = conteudo[:80].replace('\n', '↵') + ('...' if len(conteudo) > 80 else '')
    print(f"  [{tipo:7s}] {preview}")

## 5. Geração do DOCX

In [ ]:
# Criar o documento
doc = Document()

# Configurar estilo padrão
style = doc.styles["Normal"]
style.font.name = FONTE_TEXTO
style.font.size = Pt(TAMANHO_TEXTO)
style.paragraph_format.space_after = Pt(6)
style.paragraph_format.line_spacing = 1.15

# Cor cinza-escuro para fórmulas (destaque sutil)
COR_FORMULA = RGBColor(0x33, 0x33, 0x33)

# ---------------------------------------------------------
# Montar o documento a partir dos tokens
# ---------------------------------------------------------
# Após a normalização, o texto só contém \n\n para separar
# parágrafos. Quebras simples já foram convertidas em espaço.

paragrafo_atual = None

for tipo, conteudo in tokens:

    if tipo == "bloco":
        p = doc.add_paragraph()
        p.alignment = WD_ALIGN_PARAGRAPH.CENTER
        p.paragraph_format.space_before = Pt(6)
        p.paragraph_format.space_after = Pt(6)
        adicionar_run(p, conteudo, FONTE_FORMULA, TAMANHO_FORMULA, cor=COR_FORMULA)
        paragrafo_atual = None

    elif tipo == "inline":
        # Tratado igual a 'bloco': parágrafo separado, centralizado
        p = doc.add_paragraph()
        p.alignment = WD_ALIGN_PARAGRAPH.CENTER
        p.paragraph_format.space_before = Pt(6)
        p.paragraph_format.space_after = Pt(6)
        adicionar_run(p, conteudo, FONTE_FORMULA, TAMANHO_FORMULA, cor=COR_FORMULA)
        paragrafo_atual = None

    elif tipo == "italico":
        if paragrafo_atual is None:
            paragrafo_atual = doc.add_paragraph()
        adicionar_run(paragrafo_atual, conteudo, FONTE_TEXTO, TAMANHO_TEXTO, italico=True)

    elif tipo == "texto":
        # Dividir apenas por \n\n (quebras de parágrafo reais)
        paragrafos = conteudo.split('\n\n')

        for i, trecho in enumerate(paragrafos):
            if i > 0:
                paragrafo_atual = None  # forçar novo parágrafo

            trecho_limpo = trecho.strip()
            if not trecho_limpo:
                continue

            if paragrafo_atual is None:
                paragrafo_atual = doc.add_paragraph()

            adicionar_run(paragrafo_atual, trecho_limpo, FONTE_TEXTO, TAMANHO_TEXTO)


# Salvar
doc.save(ARQUIVO_SAIDA)
print(f"Documento salvo: {ARQUIVO_SAIDA}")
print(f"Total de parágrafos: {len(doc.paragraphs)}")

## 6. Pré-visualização do resultado

Mostra o conteúdo do `.docx` gerado para conferência rápida.

In [ ]:
doc_check = Document(ARQUIVO_SAIDA)

print("=" * 70)
print("PRÉ-VISUALIZAÇÃO DO DOCX")
print("=" * 70)

for i, p in enumerate(doc_check.paragraphs, 1):
    runs_info = []
    for r in p.runs:
        flags = []
        if r.font.italic:
            flags.append("itálico")
        if r.font.name == FONTE_FORMULA:
            flags.append("mono")
        flag_str = f" [{', '.join(flags)}]" if flags else ""
        runs_info.append(f"{r.text}{flag_str}")

    alinhamento = "CENTER" if p.alignment == WD_ALIGN_PARAGRAPH.CENTER else "LEFT"
    print(f"\nP{i} ({alinhamento}):")
    for info in runs_info:
        print(f"  → {info}")